# 1.5 langgraph add_message 方法
根据id进行合并消息

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

# id 用于唯一标识消息，方便后续操作。并没有定义可以是任何类型
left = [SystemMessage(content="你是一个专业的翻译", id="1"),
        HumanMessage(content="你好", id="2"),
        AIMessage(content="您好，我是一个专业的翻译", id="3")]

right = [HumanMessage(content="我是老王， 你是小王", id="2"),
         AIMessage(content="好的，我记住了", id="3"),
         HumanMessage(content="你是谁？", id="4"),
         AIMessage(content="我是小王", id="5")
         ]

merged = add_messages(left, right)
print(merged)


[SystemMessage(content='你是一个专业的翻译', additional_kwargs={}, response_metadata={}, id='1'), HumanMessage(content='我是老王， 你是小王', additional_kwargs={}, response_metadata={}, id='2'), AIMessage(content='好的，我记住了', additional_kwargs={}, response_metadata={}, id='3', tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='4'), AIMessage(content='我是小王', additional_kwargs={}, response_metadata={}, id='5', tool_calls=[], invalid_tool_calls=[])]


如果定义状态没有显示定义reducer, langgraph则会使用默认行为，即覆盖更新

In [1]:

from langgraph.graph import StateGraph, START, END
from typing import TypedDict


# 定义状态
class OverAllState(TypedDict):
    logs: list[str]
    cur_id: str


# 定义节点
def node_a(state: OverAllState) -> OverAllState:
    return {
        "logs": ["node_a"],
        "cur_id": "node_a"
    }


def node_b(state: OverAllState) -> OverAllState:
    return {
        "logs": ["node_b"],
        "cur_id": "node_b"
    }


# 创建图
builder = StateGraph(state_schema=OverAllState)
# 添加节点
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
# 添加边
builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

# 获取图
graph = builder.compile()
# 运行图
result = graph.invoke({"logs": ["START"], "id": "start"})
print('=' * 30, '-> result <-', '=' * 30)
# 结果：最后一个节点node_b覆盖了状态
print(result)

============================== -> result <- ==============================
{'logs': ['node_b'], 'cur_id': 'node_b'}
